# AV2 - Etapa 1: ingestão e tokenização
**Grupo 2 - Responsável: Diego Bispo**

Este notebook é independente das células da AV1. No Colab, use **Ambiente de
execução > Executar tudo**. Os artefatos da AV1 são apenas lidos.

- Tokenizer: tiktoken 0.14.0, `cl100k_base`, `encode_ordinary`.
- Entrada: corpus e candidatos completos da AV1 (sem corte Top-k).
- Saída: instâncias de mochila 0/1 com W = 512, 1024, 2048, 4096.
- Não executa ainda os seletores guloso/PD e não chama uma LLM.

W é o orçamento dos textos dos chunks. A integração reservará espaço adicional
para instruções, consulta, citações e resposta do modelo.


In [1]:
import json
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/diegofnf/PAA_UFS_2026_2_Bispo_Diego_Marques_Gabriel_Andrade_Laryssa_Santos_Kaio_Melo_Victor.git'
try:
    import google.colab
    NO_COLAB = True
except ImportError:
    NO_COLAB = False
REPO_DIR = Path('/content/PAA_AV2') if NO_COLAB else Path.cwd()
if not (REPO_DIR / '.git').exists():
    subprocess.run(['git', 'clone', '--branch', 'av2', REPO_URL, str(REPO_DIR)], check=True)
if not (REPO_DIR / 'av2/preparar_instancias.py').is_file():
    raise RuntimeError('Use uma copia da branch av2; a main da AV1 nao contem esta etapa.')
print('Python em execucao:', sys.executable)
print('Commit em execucao:', subprocess.check_output(
    ['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip())
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                str(REPO_DIR / 'av2/requirements.txt'), 'pytest==9.1.1', 'nltk==3.10.3'], check=True)


Python em execucao: C:\Users\Diego\Documents\Mestrado\PAA_ATIVIDADE_2\tmp\validacao-etapa1\.venv\Scripts\python.exe
Commit em execucao: b984f377c101386f27399da683bed644de38e987


CompletedProcess(args=['C:\\Users\\Diego\\Documents\\Mestrado\\PAA_ATIVIDADE_2\\tmp\\validacao-etapa1\\.venv\\Scripts\\python.exe', '-m', 'pip', 'install', '-q', '-r', 'C:\\Users\\Diego\\Documents\\Mestrado\\PAA_ATIVIDADE_2\\tmp\\validacao-etapa1\\av2\\requirements.txt', 'pytest==9.1.1', 'nltk==3.10.3'], returncode=0)

In [2]:
subprocess.run([sys.executable, str(REPO_DIR / 'av2/preparar_instancias.py')],
               cwd=REPO_DIR, check=True)
relatorio = json.loads((REPO_DIR / 'av2/dados/relatorio_preparacao.json').read_text(encoding='utf-8'))
print('Consulta:', relatorio['metadados']['consulta'])
print('Tokenizer:', relatorio['metadados']['tokenizer'])
print('Chunks:', relatorio['total_chunks'], '| Candidatos positivos:', relatorio['total_candidatos_positivos'])
for instancia in relatorio['instancias']:
    print(instancia)
print('Tempos de preparacao (s):', relatorio['tempos_segundos'])


Consulta: critérios para atribuição de bolsas e requisitos de matrícula
Tokenizer: {'biblioteca': 'tiktoken', 'versao': '0.14.0', 'encoding': 'cl100k_base', 'metodo': 'encode_ordinary', 'modelo_llm': None}
Chunks: 182 | Candidatos positivos: 75
{'id_instancia': 'W_512', 'W': 512, 'total_itens': 75, 'total_itens_acima_W': 1}
{'id_instancia': 'W_1024', 'W': 1024, 'total_itens': 75, 'total_itens_acima_W': 0}
{'id_instancia': 'W_2048', 'W': 2048, 'total_itens': 75, 'total_itens_acima_W': 0}
{'id_instancia': 'W_4096', 'W': 4096, 'total_itens': 75, 'total_itens_acima_W': 0}
Tempos de preparacao (s): {'leitura_entradas_io': 0.023072699987096712, 'validacao_associacao_ram': 0.00032160000409930944, 'carregamento_tokenizer': 0.09924640000099316, 'tokenizacao_ram': 0.02193789999000728, 'montagem_instancias_ram': 4.949999856762588e-05, 'serializacao_gravacao_dados_io': 0.006086700013838708, 'total_antes_gravar_relatorio': 0.32430729997577146}


In [3]:
subprocess.run([sys.executable, '-m', 'pytest', 'av2/test_preparar_instancias.py',
                '1_scripts/test_etapa_4.py', '-q',
                '--junitxml=av2/evidencias/testes.xml'], cwd=REPO_DIR, check=True)
instancias = json.loads((REPO_DIR / 'av2/dados/instancias.json').read_text(encoding='utf-8'))
assert [i['W'] for i in instancias['instancias']] == [512, 1024, 2048, 4096]
assert all(c['w_i'] > 0 and c['v_i'] > 0 for c in instancias['itens'])
print('Instancias prontas para as etapas de selecao. Veja av2/README.md para o contrato.')


Instancias prontas para as etapas de selecao. Veja av2/README.md para o contrato.


## Próxima etapa
Kaio e Diego podem carregar `av2/dados/instancias.json`. A lista `itens` contém
texto, origem, `id_chunk`, custo `w_i` e relevância `v_i`; cada cenário fornece
`W` e `ids_itens`. A lista não é uma seleção ótima: ambos os algoritmos devem
receber os mesmos candidatos e respeitar o orçamento.

No Colab, os resultados ficam na sessão em `/content/PAA_AV2/av2/dados/`.
Executar novamente atualiza esses arquivos locais; não faz commit nem push.
